# Compact Singh RNA inputs

This executed export preserves the exact resolved Day-7 RNA normalization and selected-gene matrix of manuscript v5. It does not refit or change the NMF. The compressed normalized RNA matrix permits independent feature selection and the dense selected-gene checkpoint permits model reproduction. The original source object stays external.

Environment: existing chronoclone at `/path/to/ChronoClone/.runtime/chronoclone`, kernel chronoclone; versions print below. Run from this directory with `CHRONOCLONE_SINGH_H5AD=/path/to/exact/adata_valid_palantir_20260626.h5ad python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=chronoclone-paper 0002_prepare_compact_rna_input.ipynb`. Without that variable, this notebook verifies the shipped checkpoints without reopening an external object. Recorded MacBook command used the exact source path shown in the output.

Executed extraction command on the source MacBook: `CHRONOCLONE_SINGH_H5AD=/path/to/ChronoClone/reference_workspace/data/adata_valid_palantir_20260626.h5ad /path/to/ChronoClone/.runtime/chronoclone/bin/python /path/to/ChronoClone/analysis_workspace/manuscript/chronoclone_panel_revision_20260910/execute_notebook.py /path/to/ChronoClone/ChronoClone_paper/analysis/singh/0002_prepare_compact_rna_input.ipynb`.

Publication verification command, run from this directory: `JUPYTER_PATH=/path/to/ChronoClone/.runtime/jupyter /path/to/ChronoClone/.runtime/chronoclone/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=chronoclone-paper 0002_prepare_compact_rna_input.ipynb`.

In [1]:
from pathlib import Path
import os,sys,json,hashlib,platform,importlib.metadata as metadata
import numpy as np,pandas as pd,scipy.sparse as sp
from IPython.display import display
HERE=Path.cwd()
OUT=HERE/'checkpoints/rna_input';OUT.mkdir(parents=True,exist_ok=True)
source=os.environ.get('CHRONOCLONE_SINGH_H5AD')
environment={'name':'chronoclone','prefix':str(Path(sys.executable).parent.parent),'kernel':'chronoclone-paper','Python':platform.python_version(),'versions':{p:metadata.version(p) for p in ['anndata','scanpy','numpy','pandas','scipy','scikit-learn']},'source':source}
display(environment)
def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for v in iter(lambda:f.read(4*1024*1024),b''):h.update(v)
    return h.hexdigest()


{'name': 'chronoclone',
 'prefix': '/path/to/ChronoClone/.runtime/chronoclone',
 'kernel': 'chronoclone-paper',
 'Python': '3.11.15',
 'versions': {'anndata': '0.12.18',
  'scanpy': '1.11.5',
  'numpy': '2.4.6',
  'pandas': '2.3.3',
  'scipy': '1.17.1',
  'scikit-learn': '1.9.0'},
 'source': None}

In [2]:
if source:
    import anndata as ad,scanpy as sc
    from sklearn.metrics import mean_squared_error
    DATA=Path(source).expanduser().resolve()
    expected='c96a765b7697e7927b3e957e2a2240fcc49fdace1403e239240a994e4cb520b6'
    assert sha(DATA)==expected,'Exact registered source hash required'
    cohort=pd.read_csv(HERE/'tables/programs/day7_resolved_family_registry.csv',index_col='cell_id',dtype={'clone':str})
    A=ad.read_h5ad(DATA,backed='r')
    genes=pd.Index(A.var_names[:20994])
    assert A.var_names[20994].startswith('Sapphire')
    X=A[cohort.index,:20994].to_memory().X.astype(np.float64)
    A.file.close()
    X.data=np.expm1(X.data)
    RNA=ad.AnnData(X=X,obs=cohort.copy(),var=pd.DataFrame(index=genes))
    sc.pp.normalize_total(RNA,target_sum=10000)
    RNA.X.data=np.log1p(RNA.X.data);RNA.uns['log1p']={'base':None}
    SEX=['Xist','Tsix','Ddx3y','Eif2s3y','Kdm5d','Usp9y','Uty']
    G=RNA[:,~(genes.str.startswith('mt-')|genes.isin(SEX))].copy()
    sc.pp.highly_variable_genes(G,n_top_genes=2000,flavor='seurat')
    V=G[:,G.var.highly_variable].copy()
    matrix=V.X.toarray().astype(np.float64);matrix[matrix<0]=0
    retained=np.load(HERE/'checkpoints/programs/retained_program_spectra.npz',allow_pickle=True)
    assert np.array_equal(V.var_names.to_numpy(dtype=str),retained['genes'].astype(str))
    assert np.array_equal(cohort.index.to_numpy(dtype=str),retained['cell_ids'].astype(str))
    error=float(np.linalg.norm(matrix-retained['W']@retained['H']))
    fit=json.loads((HERE/'provenance/programs/model_fit.json').read_text())
    assert np.isclose(error,fit['retained_reconstruction_error'],rtol=1e-10,atol=1e-8)
    sp.save_npz(OUT/'rna_log1p_matrix.npz',RNA.X,compressed=True)
    pd.DataFrame({'gene':RNA.var_names}).to_csv(OUT/'rna_genes.csv',index=False)
    cohort.to_csv(OUT/'cell_registry.csv',index_label='cell_id')
    np.savez_compressed(OUT/'nmf_input.npz',X=matrix,genes=V.var_names.to_numpy(dtype=str),cell_ids=cohort.index.to_numpy(dtype=str))
    report={'source_sha256':expected,'source_path':str(DATA),'normalization':'RNA-only total10000 thenlog1p before feature exclusions','rows':matrix.shape[0],'rna_features':RNA.n_vars,'selected_features':matrix.shape[1],'selected_gene_order_exact':True,'cell_order_exact':True,'reconstruction_error':error,'original_retained_reconstruction_error':fit['retained_reconstruction_error'],'model_refitted':False,'environment':environment}
    (OUT/'extraction_receipt.json').write_text(json.dumps(report,indent=2)+'\n')
    manifest=[{'path':str(f.relative_to(HERE)),'bytes':f.stat().st_size,'sha256':sha(f)} for f in sorted(OUT.iterdir()) if f.name!='manifest.json']
    (OUT/'manifest.json').write_text(json.dumps({'files':manifest},indent=2)+'\n')
    display(report)
else:
    print('No external source configured; verifying shipped checkpoints.')


No external source configured; verifying shipped checkpoints.


In [3]:
manifest=json.loads((OUT/'manifest.json').read_text())['files']
assert all(sha(HERE/r['path'])==r['sha256'] for r in manifest)
M=np.load(OUT/'nmf_input.npz')
R=sp.load_npz(OUT/'rna_log1p_matrix.npz')
assert M['X'].shape==(2713,2000) and R.shape==(2713,20994)
assert np.isfinite(M['X']).all() and (M['X']>=0).all()
check=R.copy();check.data=np.expm1(check.data)
assert np.allclose(np.asarray(check.sum(axis=1)).ravel(),10000,rtol=1e-8)
display(pd.DataFrame(manifest))
print('Compact RNA input integrity and normalization checks passed; no NMF refit performed.')


,path,bytes,sha256,historical_sha256
0,checkpoints/rna_input/cell_registry.csv,135935,75ee3ea5a44258dcc68d7bb00769ddeba6e5a8401e6f16...,75ee3ea5a44258dcc68d7bb00769ddeba6e5a8401e6f16...
1,checkpoints/rna_input/extraction_receipt.json,1043,3653ff0a421563a59bd58127ff089e3bb4f4eed39ea909...,7b176075015c56423443e28d01ffb5e2e5269229582484...
2,checkpoints/rna_input/nmf_input.npz,2511441,ff41b68e034de7808346b14884e587d5be4a0491c26a1a...,ff41b68e034de7808346b14884e587d5be4a0491c26a1a...
3,checkpoints/rna_input/rna_genes.csv,151436,9dc2cf7010ef922c2cce9b71027b4c2ff19ad01b423890...,9dc2cf7010ef922c2cce9b71027b4c2ff19ad01b423890...
4,checkpoints/rna_input/rna_log1p_matrix.npz,23772216,b19ea5e87bb998cc6dedc23742cd521ca10f3343b39442...,b19ea5e87bb998cc6dedc23742cd521ca10f3343b39442...


Compact RNA input integrity and normalization checks passed; no NMF refit performed.
